# Data Mining Assignment: Apache Spark Data Lake Preprocessing

**Name:** Pappala Sai Sri Chandana  
**Roll No:** LCS2023037  
**Environment:** Google Colab + PySpark  
**Dataset:** sales.csv

---

## Objective
Create a simple Data Lake structure with Bronze and Silver layers. The Bronze layer stores raw, unmodified data. The Silver layer stores preprocessed data after basic cleaning with Apache Spark.

## 1. Setup: Install PySpark and Create Data Lake Structure

In [ ]:
# Install PySpark
!pip install pyspark -q

In [ ]:
import os

# Create Data Lake folder structure
os.makedirs('data_lake/bronze', exist_ok=True)
os.makedirs('data_lake/silver/preprocessed_sales', exist_ok=True)

print("Data Lake structure created:")
for root, dirs, files in os.walk('data_lake'):
    level = root.replace('data_lake', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

In [ ]:
# Download sales.csv from the class GitHub repository into the Bronze layer
!wget -q -O data_lake/bronze/sales.csv \
  "https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv"

print("sales.csv downloaded to data_lake/bronze/")
!head -5 data_lake/bronze/sales.csv

## 2. Load and Inspect with Spark

In [ ]:
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder \
    .appName("DataLakePreprocessing") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("SparkSession created successfully.")

In [ ]:
# Read sales.csv from the Bronze layer
df_bronze = spark.read.csv(
    'data_lake/bronze/sales.csv',
    header=True,
    inferSchema=True
)

print("=" * 60)
print("SCHEMA")
print("=" * 60)
df_bronze.printSchema()

In [ ]:
print("=" * 60)
print("FIRST 10 ROWS")
print("=" * 60)
df_bronze.show(10, truncate=False)

In [ ]:
total_records = df_bronze.count()
print(f"Total record count in Bronze layer: {total_records}")

### 2.1 Identify Data Quality Issues

In [ ]:
# --- Duplicate Detection ---
print("=" * 60)
print("DUPLICATE ANALYSIS")
print("=" * 60)

total_rows = df_bronze.count()
distinct_rows = df_bronze.distinct().count()
duplicate_count = total_rows - distinct_rows

print(f"Total rows: {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Exact duplicate rows: {duplicate_count}")

In [ ]:
from pyspark.sql.functions import col, count, when, isnan, isnull, sum as spark_sum

# --- Null / Missing Value Analysis ---
print("=" * 60)
print("NULL / MISSING VALUE ANALYSIS")
print("=" * 60)

null_counts = df_bronze.select(
    [spark_sum(
        when(isnull(col(c)) | (col(c) == '') | (col(c) == 'null'), 1).otherwise(0)
    ).alias(c) for c in df_bronze.columns]
)
null_counts.show(truncate=False)

In [ ]:
# Show rows with null/missing values
from pyspark.sql.functions import lit
from functools import reduce
from operator import or_

null_condition = reduce(or_, [
    isnull(col(c)) | (col(c) == '') for c in df_bronze.columns
])
print("Rows with missing values:")
df_bronze.filter(null_condition).show(truncate=False)

In [ ]:
from pyspark.sql.functions import trim, lower, initcap

# --- Text Format Inconsistencies ---
print("=" * 60)
print("TEXT FORMAT INCONSISTENCIES")
print("=" * 60)

# Check for leading/trailing whitespace in text columns
text_cols = ['customer_name', 'product', 'category', 'payment_method', 'city', 'state']

for c in text_cols:
    whitespace_rows = df_bronze.filter(col(c) != trim(col(c)))
    ws_count = whitespace_rows.count()
    if ws_count > 0:
        print(f"\n'{c}' has {ws_count} row(s) with leading/trailing whitespace:")
        whitespace_rows.select('order_id', c).show(truncate=False)

# Check category for capitalization issues
print("\nDistinct 'category' values (check for case inconsistencies):")
df_bronze.select('category').distinct().orderBy('category').show(truncate=False)

In [ ]:
# --- Invalid Numerical Values ---
print("=" * 60)
print("INVALID NUMERICAL VALUES")
print("=" * 60)

# Quantity <= 0
invalid_qty = df_bronze.filter(col('quantity') <= 0)
print(f"\nRows with quantity <= 0: {invalid_qty.count()}")
invalid_qty.show(truncate=False)

# Unit price <= 0
invalid_price = df_bronze.filter(col('unit_price') <= 0)
print(f"\nRows with unit_price <= 0: {invalid_price.count()}")
invalid_price.show(truncate=False)

# Discount percent outside 0-100 range
invalid_discount = df_bronze.filter(
    (col('discount_percent') < 0) | (col('discount_percent') > 100)
)
print(f"\nRows with discount_percent outside [0, 100]: {invalid_discount.count()}")
invalid_discount.show(truncate=False)

In [ ]:
from pyspark.sql.functions import to_date, regexp_replace

# --- Inconsistent / Invalid Dates ---
print("=" * 60)
print("DATE VALIDATION")
print("=" * 60)

# Check the distinct date formats present
print("\nSample order_date values:")
df_bronze.select('order_id', 'order_date').show(truncate=False)

# Try parsing with yyyy-MM-dd (the dominant format)
df_date_check = df_bronze.withColumn(
    'parsed_date_iso', to_date(col('order_date'), 'yyyy-MM-dd')
).withColumn(
    'parsed_date_dmy', to_date(col('order_date'), 'dd/MM/yyyy')
)

# Rows where neither format parses successfully
unparseable = df_date_check.filter(
    isnull(col('parsed_date_iso')) & isnull(col('parsed_date_dmy'))
)
print(f"\nRows where date could not be parsed in any expected format: {unparseable.count()}")
unparseable.select('order_id', 'order_date').show(truncate=False)

# Rows where only dd/MM/yyyy format works (different format)
diff_format = df_date_check.filter(
    isnull(col('parsed_date_iso')) & col('parsed_date_dmy').isNotNull()
)
print(f"\nRows with dd/MM/yyyy format (inconsistent with majority): {diff_format.count()}")
diff_format.select('order_id', 'order_date', 'parsed_date_dmy').show(truncate=False)

df_date_check.drop('parsed_date_iso', 'parsed_date_dmy')  # cleanup temp columns

In [ ]:
# --- Check for truncated / corrupted rows ---
print("=" * 60)
print("TRUNCATED / CORRUPTED ROWS")
print("=" * 60)

# Last few rows
print("\nLast 5 rows of the dataframe:")
from pyspark.sql import Window
from pyspark.sql.functions import row_number, monotonically_increasing_id

df_with_idx = df_bronze.withColumn('_idx', monotonically_increasing_id())
total = df_with_idx.count()
df_with_idx.orderBy(col('_idx').desc()).drop('_idx').show(5, truncate=False)

# Check if any row has null order_status (last column) — indicates truncated row
truncated = df_bronze.filter(isnull(col('order_status')))
print(f"\nRows with null order_status (possibly truncated): {truncated.count()}")
truncated.show(truncate=False)

### Summary of Data Quality Issues Found

| Issue | Details |
|---|---|
| Exact Duplicate Rows | Detected via Spark `distinct()` comparison |
| Missing Values | `customer_name` and `payment_method` have nulls/empty strings |
| Text Inconsistencies | Trailing whitespace in `customer_name`; lowercase category value (`sports` vs `Sports`) |
| Invalid Numerical Values | `quantity = 0`, `unit_price = 0`, negative `discount_percent` |
| Invalid Dates | Date `31/02/2026` is impossible (Feb has max 28/29 days); inconsistent format |
| Truncated Rows | Last row appears to be cut off (incomplete data) |

## 3. Basic Preprocessing — Silver Layer

In [ ]:
from pyspark.sql.functions import (
    col, trim, initcap, when, isnull, to_date, coalesce, lit, avg, abs as spark_abs
)

# Start with the bronze data
df = df_bronze
print(f"Starting row count: {df.count()}")

### Step 1: Remove Exact Duplicate Rows

In [ ]:
before_dedup = df.count()
df = df.dropDuplicates()
after_dedup = df.count()
print(f"Rows before deduplication: {before_dedup}")
print(f"Rows after deduplication:  {after_dedup}")
print(f"Duplicates removed: {before_dedup - after_dedup}")

### Step 2: Remove Truncated / Corrupted Rows

In [ ]:
before_trunc = df.count()

# Remove rows where order_status is null (indicates truncated/corrupted row)
df = df.filter(col('order_status').isNotNull())

after_trunc = df.count()
print(f"Rows before removing truncated: {before_trunc}")
print(f"Rows after removing truncated:  {after_trunc}")
print(f"Truncated rows removed: {before_trunc - after_trunc}")

### Step 3: Handle Missing Values

In [ ]:
# Fill missing customer_name with 'Unknown'
df = df.withColumn(
    'customer_name',
    when(
        isnull(col('customer_name')) | (trim(col('customer_name')) == ''),
        lit('Unknown')
    ).otherwise(col('customer_name'))
)

# Fill missing payment_method with 'Unknown'
df = df.withColumn(
    'payment_method',
    when(
        isnull(col('payment_method')) | (trim(col('payment_method')) == ''),
        lit('Unknown')
    ).otherwise(col('payment_method'))
)

print("Missing values handled:")
print("  - customer_name: filled with 'Unknown'")
print("  - payment_method: filled with 'Unknown'")

# Verify no more nulls in these columns
null_check = df.select(
    spark_sum(when(isnull(col('customer_name')) | (col('customer_name') == ''), 1).otherwise(0)).alias('customer_name_nulls'),
    spark_sum(when(isnull(col('payment_method')) | (col('payment_method') == ''), 1).otherwise(0)).alias('payment_method_nulls')
)
null_check.show()

### Step 4: Trim Whitespace and Standardize Text Capitalization

In [ ]:
# Trim extra spaces from all text columns
text_columns = ['customer_name', 'product', 'category', 'payment_method', 'city', 'state']

for c in text_columns:
    df = df.withColumn(c, trim(col(c)))

# Standardize category capitalization — use Title Case
# This fixes 'sports' -> 'Sports', etc.
df = df.withColumn('category', initcap(col('category')))

print("Text columns trimmed and category capitalization standardized.")
print("\nDistinct category values after standardization:")
df.select('category').distinct().orderBy('category').show(truncate=False)

### Step 5: Validate Numerical Fields

In [ ]:
# --- Quantity Validation ---
# Set quantity = 0 to 1 (likely means 1 was intended since the order exists)
df = df.withColumn(
    'quantity',
    when(col('quantity') <= 0, 1).otherwise(col('quantity'))
)
print("Quantity: values <= 0 set to 1 (preserving the record)")

# --- Unit Price Validation ---
# unit_price = 0 is invalid for a real product; replace with null then fill with category median
# For simplicity, we'll set to null and then use the category average
from pyspark.sql.functions import avg as spark_avg, round as spark_round

# Calculate average unit_price per product (excluding zeros)
product_avg_price = df.filter(col('unit_price') > 0) \
    .groupBy('product') \
    .agg(spark_round(spark_avg('unit_price'), 0).alias('avg_price'))

# Join and replace zero prices
df = df.join(product_avg_price, on='product', how='left')
df = df.withColumn(
    'unit_price',
    when(col('unit_price') <= 0, col('avg_price')).otherwise(col('unit_price'))
)
df = df.drop('avg_price')
print("Unit price: values <= 0 replaced with product-level average")

# --- Discount Percentage Validation ---
# Negative discount is clearly invalid; take absolute value (likely a data entry error)
df = df.withColumn(
    'discount_percent',
    when(col('discount_percent') < 0, spark_abs(col('discount_percent')))
    .when(col('discount_percent') > 100, 100)
    .otherwise(col('discount_percent'))
)
print("Discount percent: negative values corrected to absolute; values > 100 capped at 100")

# Verify
print(f"\nQuantity range: {df.agg({'quantity': 'min'}).collect()[0][0]} - {df.agg({'quantity': 'max'}).collect()[0][0]}")
print(f"Unit price range: {df.agg({'unit_price': 'min'}).collect()[0][0]} - {df.agg({'unit_price': 'max'}).collect()[0][0]}")
print(f"Discount range: {df.agg({'discount_percent': 'min'}).collect()[0][0]} - {df.agg({'discount_percent': 'max'}).collect()[0][0]}")

### Step 6: Standardize and Validate Dates

In [ ]:
# Convert dates to consistent format (yyyy-MM-dd)
# Handle two formats: yyyy-MM-dd (majority) and dd/MM/yyyy (minority)

df = df.withColumn(
    'order_date_parsed',
    coalesce(
        to_date(col('order_date'), 'yyyy-MM-dd'),   # Try ISO format first
        to_date(col('order_date'), 'dd/MM/yyyy')    # Then try dd/MM/yyyy
    )
)

# Check how many dates could not be parsed (invalid dates like 31/02/2026)
unparsed = df.filter(isnull(col('order_date_parsed')))
print(f"Rows with unparseable/invalid dates: {unparsed.count()}")
unparsed.select('order_id', 'order_date').show(truncate=False)

# For invalid dates, set to null (preserving the record)
df = df.withColumn('order_date', col('order_date_parsed')).drop('order_date_parsed')

print("\nDates standardized to yyyy-MM-dd format.")
print("Invalid dates (e.g., 31/02/2026) set to null to preserve the record.")
df.select('order_id', 'order_date').show(5, truncate=False)

### Step 7: Reorder Columns and Final Check

In [ ]:
# Ensure proper column order
column_order = [
    'order_id', 'customer_id', 'customer_name', 'product', 'category',
    'quantity', 'unit_price', 'discount_percent', 'payment_method',
    'city', 'state', 'order_date', 'order_status'
]
df = df.select(column_order)

print("Final Silver DataFrame Schema:")
df.printSchema()
print(f"\nFinal Silver row count: {df.count()}")
df.show(10, truncate=False)

### Save Silver Layer

In [ ]:
# Save as CSV to the Silver folder
df.coalesce(1).write.csv(
    'data_lake/silver/preprocessed_sales',
    header=True,
    mode='overwrite'
)

print("Silver layer saved to data_lake/silver/preprocessed_sales/")

# Also save a single merged CSV for easy submission
df.toPandas().to_csv('data_lake/silver/sales_silver.csv', index=False)
print("Combined Silver CSV saved to data_lake/silver/sales_silver.csv")

## 4. Verification

In [ ]:
# --- Row Count Before and After ---
print("=" * 60)
print("ROW COUNT COMPARISON")
print("=" * 60)
print(f"Bronze (raw) row count:         {total_records}")
print(f"Silver (preprocessed) row count: {df.count()}")
print(f"Rows removed:                   {total_records - df.count()}")

In [ ]:
# --- Read Silver Data Back with Spark ---
print("=" * 60)
print("READING SILVER DATA BACK")
print("=" * 60)

df_silver = spark.read.csv(
    'data_lake/silver/sales_silver.csv',
    header=True,
    inferSchema=True
)

print(f"Silver data row count: {df_silver.count()}")
print("\nSilver data schema:")
df_silver.printSchema()
print("\nFirst 10 rows of Silver data:")
df_silver.show(10, truncate=False)

In [ ]:
# --- Final Summary of Preprocessing ---
print("=" * 60)
print("PREPROCESSING SUMMARY")
print("=" * 60)
print("""
Preprocessing operations performed:

1. DUPLICATE REMOVAL:
   - Removed exact duplicate rows using dropDuplicates().

2. TRUNCATED ROW REMOVAL:
   - Removed rows with null order_status (incomplete/corrupted data).

3. MISSING VALUE HANDLING:
   - customer_name: Filled empty/null values with 'Unknown'.
   - payment_method: Filled empty/null values with 'Unknown'.

4. TEXT STANDARDIZATION:
   - Trimmed leading/trailing whitespace from all text columns.
   - Standardized category capitalization using Title Case
     (e.g., 'sports' -> 'Sports').

5. NUMERICAL VALIDATION:
   - quantity <= 0: Set to 1 (preserving records).
   - unit_price <= 0: Replaced with product-level average price.
   - discount_percent < 0: Converted to absolute value.
   - discount_percent > 100: Capped at 100.

6. DATE STANDARDIZATION:
   - Converted all valid dates to yyyy-MM-dd (DateType) format.
   - Handled dd/MM/yyyy format dates by parsing with coalesce.
   - Invalid dates (e.g., 31/02/2026) set to null to preserve records.
""")

In [ ]:
# Data Lake structure after processing
print("=" * 60)
print("FINAL DATA LAKE STRUCTURE")
print("=" * 60)
for root, dirs, files in os.walk('data_lake'):
    level = root.replace('data_lake', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        print(f'{subindent}{file} ({size:,} bytes)')

In [ ]:
# Stop SparkSession
spark.stop()
print("SparkSession stopped. Notebook complete.")